In [1]:
import requests
from bs4 import BeautifulSoup

# Fetch the homepage
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'he-IL,he;q=0.9,en-US;q=0.8,en;q=0.7'
}

response = requests.get('https://leroy.co.il', headers=headers, timeout=30)
response.encoding = 'utf-8'
soup = BeautifulSoup(response.text, 'html.parser')

print("=== PAGE TITLE ===")
print(soup.title.string if soup.title else "No title")

print("\n=== META TAGS ===")
for meta in soup.find_all('meta'):
    print(meta.attrs)

print("\n=== CSS LINKS ===")
for link in soup.find_all('link', rel='stylesheet'):
    print(link.get('href', 'N/A'))

print("\n=== INLINE STYLES (first 5) ===")
styles = soup.find_all('style')
for i, style in enumerate(styles[:5]):
    text = style.string or style.get_text()
    if text:
        print(f"--- Style block {i+1} (first 500 chars) ---")
        print(text[:500])

print("\n=== NAVIGATION STRUCTURE ===")
navs = soup.find_all('nav')
for nav in navs:
    print(f"Nav class: {nav.get('class')}")
    links = nav.find_all('a')
    for a in links[:20]:
        print(f"  - {a.get_text(strip=True)}: {a.get('href', 'N/A')}")

print("\n=== HEADER ===")
headers_el = soup.find_all('header')
for h in headers_el:
    print(f"Header class: {h.get('class')}")
    links = h.find_all('a')
    for a in links[:15]:
        print(f"  - {a.get_text(strip=True)}: {a.get('href', 'N/A')}")

print("\n=== MAIN SECTIONS (by section/div with id or class) ===")
sections = soup.find_all('section')
for sec in sections[:20]:
    cls = sec.get('class', [])
    sid = sec.get('id', '')
    heading = sec.find(['h1', 'h2', 'h3'])
    heading_text = heading.get_text(strip=True) if heading else ''
    print(f"Section id={sid}, class={cls}, heading='{heading_text}'")

print("\n=== ALL HEADINGS ===")
for tag in ['h1', 'h2', 'h3', 'h4']:
    headings = soup.find_all(tag)
    if headings:
        print(f"\n{tag.upper()}:")
        for h in headings[:10]:
            print(f"  - {h.get_text(strip=True)}")

print("\n=== FOOTER ===")
footers = soup.find_all('footer')
for f in footers:
    print(f"Footer class: {f.get('class')}")
    links = f.find_all('a')
    for a in links[:20]:
        print(f"  - {a.get_text(strip=True)}: {a.get('href', 'N/A')}")

print("\n=== IMAGES (first 20) ===")
imgs = soup.find_all('img')
for img in imgs[:20]:
    print(f"  src={img.get('src', 'N/A')[:100]}, alt={img.get('alt', 'N/A')}")

print("\n=== SCRIPTS (external, first 10) ===")
scripts = soup.find_all('script', src=True)
for s in scripts[:10]:
    print(f"  {s.get('src', '')[:100]}")

# Look for color-related CSS
print("\n=== BODY/ROOT CLASSES ===")
body = soup.find('body')
if body:
    print(f"Body class: {body.get('class')}")
    print(f"Body id: {body.get('id')}")

# Check for any data attributes or framework hints
print("\n=== FRAMEWORK HINTS ===")
divs_with_id = soup.find_all('div', id=True)
for d in divs_with_id[:10]:
    print(f"  div id={d.get('id')}, class={str(d.get('class', ''))[:80]}")

print(f"\n=== TOTAL HTML LENGTH: {len(response.text)} chars ===")

=== PAGE TITLE ===
לירוי דיזיין - בית של עיצובים מגוון רחב של מיטות ומזרנים בעיצוב מושלם!

=== META TAGS ===
{'charset': 'UTF-8'}
{'name': 'viewport', 'content': 'width=device-width, initial-scale=1'}
{'name': 'robots', 'content': 'index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1'}
{'name': 'description', 'content': 'לחברתנו מגוון רחב של מיטות מכל הסוגים: מיטות זוגיות, מיטה יהודית, מיטה וחצי, מיטות נוער, ספות ועוד הכוללים מזרנים לכל סוגי המיטות.'}
{'property': 'og:locale', 'content': 'he_IL'}
{'property': 'og:type', 'content': 'website'}
{'property': 'og:title', 'content': 'לירוי דיזיין - בית של עיצובים מגוון רחב של מיטות ומזרנים בעיצוב מושלם!'}
{'property': 'og:description', 'content': 'לחברתנו מגוון רחב של מיטות מכל הסוגים: מיטות זוגיות, מיטה יהודית, מיטה וחצי, מיטות נוער, ספות ועוד הכוללים מזרנים לכל סוגי המיטות.'}
{'property': 'og:url', 'content': 'https://leroy.co.il/'}
{'property': 'og:site_name', 'content': 'Leroy designs'}
{'property': 'article:publi

In [2]:
import requests
from bs4 import BeautifulSoup
import re

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'he-IL,he;q=0.9,en-US;q=0.8,en;q=0.7'
}

response = requests.get('https://leroy.co.il', headers=headers, timeout=30)
response.encoding = 'utf-8'
soup = BeautifulSoup(response.text, 'html.parser')

# Extract ALL inline styles to find colors and fonts
print("=== ALL INLINE STYLES - COLOR & FONT ANALYSIS ===")
styles = soup.find_all('style')
all_css = ""
for style in styles:
    text = style.string or style.get_text()
    if text:
        all_css += text + "\n"

# Extract colors
colors = set(re.findall(r'#[0-9a-fA-F]{3,8}', all_css))
print(f"\nColors found ({len(colors)}):")
for c in sorted(colors):
    print(f"  {c}")

# Extract rgb/rgba colors
rgb_colors = set(re.findall(r'rgba?\([^)]+\)', all_css))
print(f"\nRGB/RGBA colors ({len(rgb_colors)}):")
for c in sorted(rgb_colors):
    print(f"  {c}")

# Extract font families
fonts = set(re.findall(r'font-family\s*:\s*([^;}{]+)', all_css))
print(f"\nFont families ({len(fonts)}):")
for f in sorted(fonts):
    print(f"  {f.strip()}")

# Extract Google Fonts
gfonts = soup.find_all('link', href=re.compile(r'fonts\.google'))
print(f"\nGoogle Fonts links:")
for gf in gfonts:
    print(f"  {gf.get('href', '')[:200]}")

# Look for background-color patterns
bg_colors = set(re.findall(r'background-color\s*:\s*([^;}{]+)', all_css))
print(f"\nBackground colors:")
for c in sorted(bg_colors):
    print(f"  {c.strip()}")

# Extract Elementor widget types used
print("\n=== ELEMENTOR WIDGETS USED ===")
elementor_widgets = set()
for el in soup.find_all(class_=re.compile(r'elementor-widget-')):
    classes = el.get('class', [])
    for cls in classes:
        if cls.startswith('elementor-widget-'):
            elementor_widgets.add(cls)
for w in sorted(elementor_widgets):
    print(f"  {w}")

# Extract all sections with their Elementor data
print("\n=== ELEMENTOR SECTIONS DETAILED ===")
sections = soup.find_all('section', class_=re.compile(r'elementor-section'))
for i, sec in enumerate(sections[:30]):
    cls = ' '.join(sec.get('class', []))
    data_id = sec.get('data-id', '')
    # Find headings in this section
    headings = sec.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'], recursive=True)
    heading_texts = [h.get_text(strip=True) for h in headings[:3]]
    # Find images
    imgs = sec.find_all('img', recursive=True)
    img_alts = [img.get('alt', '') for img in imgs[:3]]
    # Find links/buttons
    buttons = sec.find_all('a', class_=re.compile(r'elementor-button'))
    btn_texts = [b.get_text(strip=True) for b in buttons[:3]]
    
    print(f"\nSection {i+1} (data-id={data_id}):")
    if heading_texts:
        print(f"  Headings: {heading_texts}")
    if img_alts:
        print(f"  Images: {img_alts}")
    if btn_texts:
        print(f"  Buttons: {btn_texts}")

# Extract product-related elements
print("\n=== PRODUCT CARDS / WOO ELEMENTS ===")
products = soup.find_all(class_=re.compile(r'product'))
print(f"Product-related elements: {len(products)}")
for p in products[:10]:
    cls = ' '.join(p.get('class', []))
    title = p.find(['h2', 'h3', 'h4'])
    price = p.find(class_=re.compile(r'price'))
    print(f"  Class: {cls[:80]}")
    if title:
        print(f"    Title: {title.get_text(strip=True)}")
    if price:
        print(f"    Price: {price.get_text(strip=True)}")

# Extract all text content for understanding the page
print("\n=== KEY TEXT CONTENT ===")
# Get all paragraphs
paragraphs = soup.find_all('p')
for p in paragraphs[:15]:
    text = p.get_text(strip=True)
    if text and len(text) > 10:
        print(f"  {text[:150]}")

# Extract all links for site structure
print("\n=== ALL UNIQUE INTERNAL LINKS ===")
all_links = set()
for a in soup.find_all('a', href=True):
    href = a.get('href', '')
    if 'leroy.co.il' in href or href.startswith('/'):
        all_links.add(href)
for link in sorted(all_links):
    print(f"  {link}")

# Extract phone numbers, addresses, social links
print("\n=== CONTACT INFO ===")
# Phone
phones = re.findall(r'[\d\-]{9,12}', soup.get_text())
print(f"Phone numbers: {set(phones)}")

# Social links
social = soup.find_all('a', href=re.compile(r'facebook|instagram|twitter|youtube|whatsapp|tiktok'))
for s in social:
    print(f"  Social: {s.get('href', '')[:100]}")

# WhatsApp / Chat widgets
print("\n=== CHAT/CONTACT WIDGETS ===")
chaty = soup.find_all(class_=re.compile(r'chaty'))
print(f"Chaty elements: {len(chaty)}")

# Accessibility
print("\n=== ACCESSIBILITY ===")
accessibility = soup.find_all(class_=re.compile(r'accessibility|pojo'))
print(f"Accessibility elements: {len(accessibility)}")

# Forms
print("\n=== FORMS ===")
forms = soup.find_all('form')
for form in forms:
    print(f"  Form action={form.get('action', 'N/A')[:80]}, method={form.get('method', 'N/A')}")
    inputs = form.find_all('input')
    for inp in inputs[:5]:
        print(f"    Input: name={inp.get('name', 'N/A')}, type={inp.get('type', 'N/A')}, placeholder={inp.get('placeholder', 'N/A')}")

# All images for media analysis
print("\n=== ALL IMAGES ===")
all_imgs = soup.find_all('img')
print(f"Total images: {len(all_imgs)}")
for img in all_imgs:
    src = img.get('src', img.get('data-src', 'N/A'))
    alt = img.get('alt', 'N/A')
    if src and src != 'N/A':
        print(f"  {alt}: {src[:120]}")

=== ALL INLINE STYLES - COLOR & FONT ANALYSIS ===

Colors found (67):
  #000
  #000000
  #00000000
  #00000057
  #000000DE
  #00d084
  #0170B9
  #02010100
  #0693e3
  #0A090C
  #2A2A2A
  #32373c
  #333333
  #3a3a3a
  #40464d
  #4054b2
  #424141
  #424242
  #4B4F58
  #515151
  #54595F
  #555d66
  #61CE70
  #6EC1E4
  #7A7A7A
  #7a7a7a
  #7bdcb5
  #8ed1fc
  #9b51e0
  #B69863
  #B69864
  #BEBCBC
  #C5C5C5
  #DBDBDB
  #E5E5E5
  #E6E6E6
  #E9C788
  #EAEAEA
  #EBCC98
  #EEEEEE
  #EFEFEF
  #F5F5F5
  #FAFAFA
  #FF0000
  #FFBC7D
  #FFFFFF
  #FFFFFF00
  #abb8c3
  #b69864
  #ccc
  #cf2e2e
  #dad8da
  #dddddd
  #e6e6e6
  #e7e7e7
  #eaeaea
  #ebcc98
  #eeeeee
  #f78da7
  #fafafa
  #fbfbfb
  #fcb900
  #ff0798
  #ff0b0b
  #ff6900
  #fff
  #ffffff

RGB/RGBA colors (42):
  rgb(0 0 0 / .8)
  rgb(0, 0, 0)
  rgb(0,208,130)
  rgb(107,0,62)
  rgb(113,206,126)
  rgb(122,220,180)
  rgb(151,120,209)
  rgb(152,150,240)
  rgb(155,81,224)
  rgb(169,184,195)
  rgb(182,227,212)
  rgb(199,81,192)
  rgb(2,3,129)
  rgb